In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ICE-SHEET LEVEL-2 TRAJECTORY ICL TRAINING
=========================================

Canonical dataset
-----------------
Expected dataset generated by:
    generate_ice_level2_Abeta_time_no_fixed_split.py

Default Kaggle input root:
    /kaggle/input/notebooks/xli46913/ice-sheet-vars-dataset

Physical system/operator:
    operator_j = (A_j, beta_j(x,y))

Each operator has one LF/HF trajectory with complete snapshots at:
    t = 1, 2, ..., 50

Split:
    70 train systems
    30 completely held-out IN-DISTRIBUTION test systems

There is NO temporal OOD protocol in this script.

ICL episode
-----------
For a single chosen operator/system j:

    demo_i = [XY, t_i, Uc(t_i), Uf(t_i)]
    query  = [XY, t_q, Uc(t_q)]
    target = Uf(t_q)

A, beta, and H are NOT model inputs.

Training:
    K ~ Uniform{1,...,5}
    K demo times + 1 query time are sampled without replacement
    from the SAME train-system trajectory.

Official held-out-system evaluation:
    K = 5
    For EACH held-out system, a seeded random set of 5 support times
    is generated by THIS training script (not stored in the dataset).
    Different held-out systems generally receive different support times.
    The remaining 45 time snapshots are queries.
    Total official queries = 30 * 45 = 1350.

This is NOT forecasting: demonstrations may occur before or after a query time.
The only generalization axis here is the held-out physical system (A,beta),
drawn from the same parameter/distribution family as training systems.

Model
-----
Row-sequence Transformer:
    [XY,t,Uc,Uf] x K + [XY,t,Uc] -> Uf_q

Encoders:
    separate XY encoder
    separate scalar-time encoder
    one SHARED velocity encoder for Uc/Uf

Architecture:
    D_MODEL=512
    N_HEAD=8
    N_LAYERS=5
    D_FF=1536
    dropout=0.10

Optimization:
    AdamW
    180 epochs x 600 steps = 108,000 updates
    batch size 64
    warmup + cosine LR
    direct full-field Uf target
    balanced normalized u/v MSE
    no residual/delta target
    final fixed-budget checkpoint is the official model

Scientific safeguards
---------------------
* Dataset filename + metadata + array shapes are audited.
* Dataset SHA256 is printed and stored in checkpoints.
* Normalization uses TRAIN SYSTEMS ONLY.
* Test systems are never used for normalization.
* No test-based early stopping or checkpoint selection.
* Held-out test diagnostics are OFF during training by default.
* Final held-out result uses one fixed seeded-random support partition.
* The support partition is saved separately and inside checkpoints.

Optional environment overrides
------------------------------
    BATCH_SIZE=32
    EPOCHS=180
    STEPS_PER_EPOCH=600
    RUN_TEST_DIAGNOSTICS=1
    NPZ_PATH_FIXED=/exact/path/master.npz
    EXPECTED_DATASET_SHA256=<hash>
    RESUME_FROM=/path/to/epoch060_resume.pt
"""

from __future__ import annotations

import glob
import hashlib
import json
import os
import random
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# =============================================================================
# CONFIG HELPERS
# =============================================================================

def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, str(default)))


def env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, str(default)))


SEED = env_int("SEED", 42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

INPUT_ROOT = os.environ.get(
    "INPUT_ROOT",
    "/kaggle/input/notebooks/xli46913/ice-sheet-vars-dataset",
).strip()

DATASET_FILENAME = os.environ.get(
    "DATASET_FILENAME",
    "ice_icl_Abeta100_time1to50_fullfield_noFixedTimeSplit_master.npz",
).strip()

NPZ_PATH_FIXED = os.environ.get("NPZ_PATH_FIXED", "").strip()
EXPECTED_DATASET_SHA256 = os.environ.get(
    "EXPECTED_DATASET_SHA256", ""
).strip().lower()

RUN_TAG = os.environ.get(
    "RUN_TAG",
    "ICE_ICL_Abeta100_time1to50_70train30test_"
    "XYtUcUf_sharedVelEnc_d512_L5_FF1536_K1to5_v1_ampfix",
).strip()

SAVE_DIR = os.environ.get(
    "SAVE_DIR",
    (
        f"/kaggle/working/{RUN_TAG}"
        if os.path.isdir("/kaggle/working")
        else f"./{RUN_TAG}"
    ),
).strip()
os.makedirs(SAVE_DIR, exist_ok=True)

RESUME_FROM = os.environ.get("RESUME_FROM", "").strip()

# Expected canonical dataset protocol.
N_OPERATORS_EXPECTED = 100
N_TRAIN_OPERATORS_EXPECTED = 70
N_TEST_OPERATORS_EXPECTED = 30
N_TIMES_EXPECTED = 50
LF_SIDE_EXPECTED = 15
HF_SIDE_EXPECTED = 41
N_COMP = 2

# ICL context.
MIN_DEMOS = env_int("MIN_DEMOS", 1)
MAX_DEMOS = env_int("MAX_DEMOS", 5)
TEST_K = env_int("TEST_K", 5)
TEST_SUPPORT_SEED = env_int("TEST_SUPPORT_SEED", SEED + 300)

# Training budget.
BATCH_SIZE = env_int("BATCH_SIZE", 64)
EPOCHS = env_int("EPOCHS", 180)
STEPS_PER_EPOCH = env_int("STEPS_PER_EPOCH", 600)
PRINT_EVERY = env_int("PRINT_EVERY", 5)
N_TRAIN_EVAL_CASES = env_int("N_TRAIN_EVAL_CASES", 512)
EVAL_BATCH_SIZE = env_int("EVAL_BATCH_SIZE", 64)

# Optimizer / scheduler.
WEIGHT_DECAY = env_float("WEIGHT_DECAY", 2e-4)
BASE_LR = env_float("BASE_LR", 1.5e-5)
MAX_LR = env_float("MAX_LR", 5.0e-5)
MIN_LR = env_float("MIN_LR", 1.0e-6)
WARMUP_FRAC = env_float("WARMUP_FRAC", 0.08)
GRAD_CLIP = env_float("GRAD_CLIP", 1.0)

# Model.
D_MODEL = env_int("D_MODEL", 512)
N_HEAD = env_int("N_HEAD", 8)
N_LAYERS = env_int("N_LAYERS", 5)
D_FF = env_int("D_FF", 1536)
DROPOUT = env_float("DROPOUT", 0.10)

# Output / diagnostics.
SAVE_PERIODIC_EVERY = env_int("SAVE_PERIODIC_EVERY", 30)
RUN_TEST_DIAGNOSTICS = bool(env_int("RUN_TEST_DIAGNOSTICS", 0))
TEST_DIAGNOSTIC_EVERY = env_int("TEST_DIAGNOSTIC_EVERY", 30)

USE_AMP = DEVICE == "cuda"
EPS = 1.0e-8

if not (1 <= MIN_DEMOS <= MAX_DEMOS <= 5):
    raise ValueError("Require 1 <= MIN_DEMOS <= MAX_DEMOS <= 5.")
if not (1 <= TEST_K < N_TIMES_EXPECTED):
    raise ValueError("TEST_K must be in [1, 49].")


# =============================================================================
# ROW TYPES
# =============================================================================

TYPE_XY_DEMO = 0
TYPE_TIME_DEMO = 1
TYPE_UC_DEMO = 2
TYPE_UF_DEMO = 3

TYPE_XY_QUERY = 4
TYPE_TIME_QUERY = 5
TYPE_UC_QUERY = 6

N_ROW_TYPES = 7
MAX_CTX_ROWS = 4 * MAX_DEMOS + 3


# =============================================================================
# REPRODUCIBILITY
# =============================================================================

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

rng_train = np.random.default_rng(SEED + 100)


# =============================================================================
# DATASET RESOLUTION + HASH
# =============================================================================

def sha256_file(path: str, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def resolve_dataset_path() -> str:
    if NPZ_PATH_FIXED:
        p = Path(NPZ_PATH_FIXED)
        if not p.is_file():
            raise FileNotFoundError(
                f"NPZ_PATH_FIXED is not a file: {p}"
            )
        return str(p)

    root = Path(INPUT_ROOT)
    if not root.exists():
        raise FileNotFoundError(
            f"INPUT_ROOT does not exist: {INPUT_ROOT}"
        )

    exact = sorted(
        set(
            glob.glob(
                str(root / "**" / DATASET_FILENAME),
                recursive=True,
            )
        )
    )

    if len(exact) == 1:
        return exact[0]

    if len(exact) > 1:
        raise RuntimeError(
            "Multiple exact dataset copies found. "
            "Set NPZ_PATH_FIXED explicitly:\n  "
            + "\n  ".join(exact)
        )

    # Recovery path: inspect master NPZ candidates under the supplied folder.
    candidates = sorted(
        set(
            glob.glob(
                str(root / "**" / "*master.npz"),
                recursive=True,
            )
        )
    )

    valid = []
    for p in candidates:
        try:
            with np.load(p, allow_pickle=False) as z:
                keys = set(z.files)
                needed = {
                    "time_values",
                    "A_rel_values",
                    "train_operator_indices",
                    "test_operator_indices",
                    "U_lf",
                    "U_hf",
                    "metadata_json",
                }
                if not needed.issubset(keys):
                    continue

                meta = json.loads(
                    str(np.asarray(z["metadata_json"]).reshape(()))
                )
                version = str(meta.get("version", ""))
                if (
                    "Abeta100_time1to50" in version
                    and int(meta.get("n_operators", -1)) == 100
                    and int(meta.get("n_train_operators", -1)) == 70
                    and bool(meta.get("episode_partition_in_dataset", True))
                    is False
                ):
                    valid.append(p)
        except Exception:
            pass

    if len(valid) == 1:
        return valid[0]

    if not valid:
        raise FileNotFoundError(
            f"Could not find canonical dataset '{DATASET_FILENAME}' "
            f"under {INPUT_ROOT}.\n"
            "No uniquely valid fallback *master.npz was found.\n"
            "Set NPZ_PATH_FIXED to the exact master file."
        )

    raise RuntimeError(
        "Multiple valid trajectory dataset masters found. "
        "Set NPZ_PATH_FIXED explicitly:\n  "
        + "\n  ".join(valid)
    )


NPZ_PATH = resolve_dataset_path()
DATASET_SHA256 = sha256_file(NPZ_PATH)

if EXPECTED_DATASET_SHA256:
    if DATASET_SHA256.lower() != EXPECTED_DATASET_SHA256:
        raise RuntimeError(
            "Dataset SHA256 mismatch.\n"
            f"Expected: {EXPECTED_DATASET_SHA256}\n"
            f"Actual  : {DATASET_SHA256}"
        )


print("=" * 128)
print("ICE-SHEET LEVEL-2 ICL | (A,beta) SYSTEMS x TIME | NO TEMPORAL OOD")
print("=" * 128)
print("Device                    :", DEVICE)
print("Dataset root              :", INPUT_ROOT)
print("Dataset                   :", NPZ_PATH)
print("Dataset SHA256            :", DATASET_SHA256)
print("Save dir                  :", SAVE_DIR)
print("System/operator           : (A, beta)")
print("Within-system axis        : time t=1..50")
print("Model sequence            : [XY,t,Uc,Uf]xK + [XY,t,Uc] -> Uf_q")
print("A / beta / H model input  : NO / NO / NO")
print("Temporal OOD              : NO")
print("Official checkpoint       : FINAL FIXED-BUDGET ONLY")
print("=" * 128)


# =============================================================================
# LOAD + AUDIT MASTER DATASET
# =============================================================================

data = np.load(NPZ_PATH, allow_pickle=False)

required_keys = [
    "x_lf",
    "y_lf",
    "x_hf",
    "y_hf",
    "time_values",
    "A_rel_values",
    "A_values",
    "beta_lf",
    "beta_hf",
    "train_operator_indices",
    "test_operator_indices",
    "operator_split_labels",
    "H_lf",
    "U_lf",
    "H_hf",
    "U_hf",
    "metadata_json",
]
for key in required_keys:
    if key not in data.files:
        raise KeyError(f"Missing required dataset key: {key}")

try:
    metadata = json.loads(
        str(np.asarray(data["metadata_json"]).reshape(()))
    )
except Exception as e:
    raise RuntimeError(
        f"Could not parse metadata_json: {e}"
    ) from e

expected_version = (
    "ICE_LEVEL2_Abeta100_time1to50_fullfield_noFixedTimeSplit_v2"
)
if str(metadata.get("version", "")) != expected_version:
    raise RuntimeError(
        "Wrong dataset version.\n"
        f"Expected: {expected_version}\n"
        f"Found   : {metadata.get('version')}"
    )

if int(metadata.get("n_operators", -1)) != N_OPERATORS_EXPECTED:
    raise RuntimeError("Metadata n_operators mismatch.")
if int(metadata.get("n_train_operators", -1)) != N_TRAIN_OPERATORS_EXPECTED:
    raise RuntimeError("Metadata n_train_operators mismatch.")
if int(metadata.get("n_saved_times", -1)) != N_TIMES_EXPECTED:
    raise RuntimeError("Metadata n_saved_times mismatch.")
if bool(metadata.get("episode_partition_in_dataset", True)):
    raise RuntimeError(
        "This training code expects NO dataset-fixed demo/query time partition."
    )
if bool(metadata.get("time_model_input", False)) is not True:
    raise RuntimeError("Expected time_model_input=True in metadata.")
if bool(metadata.get("beta_model_input", True)) is not False:
    raise RuntimeError("Expected beta_model_input=False in metadata.")
if bool(metadata.get("H_model_input", True)) is not False:
    raise RuntimeError("Expected H_model_input=False in metadata.")

gp_scale = float(metadata.get("beta_gp_scale", float("nan")))
if not np.isfinite(gp_scale) or abs(gp_scale - 0.35) > 1.0e-12:
    raise RuntimeError(
        f"Expected beta_gp_scale=0.35, found {gp_scale}"
    )

x_lf = np.asarray(data["x_lf"], dtype=np.float32).reshape(-1)
y_lf = np.asarray(data["y_lf"], dtype=np.float32).reshape(-1)
x_hf = np.asarray(data["x_hf"], dtype=np.float32).reshape(-1)
y_hf = np.asarray(data["y_hf"], dtype=np.float32).reshape(-1)
time_values = np.asarray(
    data["time_values"], dtype=np.float32
).reshape(-1)

A_rel_values = np.asarray(
    data["A_rel_values"], dtype=np.float32
).reshape(-1)
A_values = np.asarray(
    data["A_values"], dtype=np.float64
).reshape(-1)

train_operator_indices = np.asarray(
    data["train_operator_indices"], dtype=np.int64
).reshape(-1)
test_operator_indices = np.asarray(
    data["test_operator_indices"], dtype=np.int64
).reshape(-1)

# We need only U arrays for training. H/beta remain hidden model metadata.
U_lf_all = np.asarray(data["U_lf"], dtype=np.float32)
U_hf_all = np.asarray(data["U_hf"], dtype=np.float32)

# Shape audit of hidden fields without retaining them.
_beta_lf_shape = data["beta_lf"].shape
_beta_hf_shape = data["beta_hf"].shape
_H_lf_shape = data["H_lf"].shape
_H_hf_shape = data["H_hf"].shape

assert x_lf.size == LF_SIDE_EXPECTED
assert y_lf.size == LF_SIDE_EXPECTED
assert x_hf.size == HF_SIDE_EXPECTED
assert y_hf.size == HF_SIDE_EXPECTED

assert time_values.shape == (N_TIMES_EXPECTED,)
if not np.array_equal(
    time_values,
    np.arange(1, N_TIMES_EXPECTED + 1, dtype=np.float32),
):
    raise RuntimeError(
        "time_values must be exactly [1,2,...,50]."
    )

assert A_rel_values.shape == (N_OPERATORS_EXPECTED,)
assert A_values.shape == (N_OPERATORS_EXPECTED,)
assert train_operator_indices.shape == (N_TRAIN_OPERATORS_EXPECTED,)
assert test_operator_indices.shape == (N_TEST_OPERATORS_EXPECTED,)

assert _beta_lf_shape == (
    N_OPERATORS_EXPECTED,
    LF_SIDE_EXPECTED,
    LF_SIDE_EXPECTED,
)
assert _beta_hf_shape == (
    N_OPERATORS_EXPECTED,
    HF_SIDE_EXPECTED,
    HF_SIDE_EXPECTED,
)
assert _H_lf_shape == (
    N_OPERATORS_EXPECTED,
    N_TIMES_EXPECTED,
    LF_SIDE_EXPECTED,
    LF_SIDE_EXPECTED,
)
assert _H_hf_shape == (
    N_OPERATORS_EXPECTED,
    N_TIMES_EXPECTED,
    HF_SIDE_EXPECTED,
    HF_SIDE_EXPECTED,
)
assert U_lf_all.shape == (
    N_OPERATORS_EXPECTED,
    N_TIMES_EXPECTED,
    N_COMP,
    LF_SIDE_EXPECTED,
    LF_SIDE_EXPECTED,
)
assert U_hf_all.shape == (
    N_OPERATORS_EXPECTED,
    N_TIMES_EXPECTED,
    N_COMP,
    HF_SIDE_EXPECTED,
    HF_SIDE_EXPECTED,
)

train_set = set(map(int, train_operator_indices.tolist()))
test_set = set(map(int, test_operator_indices.tolist()))
if train_set & test_set:
    raise RuntimeError("Train/test system sets overlap.")
if len(train_set | test_set) != N_OPERATORS_EXPECTED:
    raise RuntimeError(
        "Train/test split does not cover all 100 systems."
    )

if not np.isfinite(U_lf_all).all():
    raise RuntimeError("Non-finite values in U_lf.")
if not np.isfinite(U_hf_all).all():
    raise RuntimeError("Non-finite values in U_hf.")

print("\n[dataset audit]")
print("  metadata version             :", metadata["version"])
print("  beta_gp_scale               :", gp_scale)
print("  train/test systems          :", len(train_operator_indices), "/", len(test_operator_indices))
print("  train system indices        :", train_operator_indices.tolist())
print("  test system indices         :", test_operator_indices.tolist())
print("  A_rel range                 :", float(A_rel_values.min()), "to", float(A_rel_values.max()))
print("  time axis                   :", f"{time_values[0]:.0f}..{time_values[-1]:.0f}", f"({len(time_values)} snapshots)")
print("  beta LF/HF                  :", _beta_lf_shape, "/", _beta_hf_shape)
print("  H LF/HF                     :", _H_lf_shape, "/", _H_hf_shape)
print("  U LF/HF                     :", U_lf_all.shape, "/", U_hf_all.shape)
print("  dataset-fixed time split    : NO")
print("  temporal OOD                : NO")


# =============================================================================
# LF 15x15 -> HF 41x41 UPSAMPLING
# =============================================================================

def bilinear_upsample_velocity_all(
    arr_lf: np.ndarray,
) -> np.ndarray:
    """
    [O,T,2,15,15] -> [O,T,2,41,41]
    Physical units are preserved.
    """
    O, T, comp, h, w = arr_lf.shape
    assert comp == 2
    assert h == LF_SIDE_EXPECTED
    assert w == LF_SIDE_EXPECTED

    flat = arr_lf.reshape(O * T, comp, h, w)
    out = np.empty(
        (
            O * T,
            comp,
            HF_SIDE_EXPECTED,
            HF_SIDE_EXPECTED,
        ),
        dtype=np.float32,
    )

    chunk = 512
    for s in range(0, O * T, chunk):
        e = min(O * T, s + chunk)
        x = torch.from_numpy(flat[s:e])
        y = F.interpolate(
            x,
            size=(HF_SIDE_EXPECTED, HF_SIDE_EXPECTED),
            mode="bilinear",
            align_corners=True,
        )
        out[s:e] = y.numpy().astype(
            np.float32, copy=False
        )

    return out.reshape(
        O,
        T,
        comp,
        HF_SIDE_EXPECTED,
        HF_SIDE_EXPECTED,
    )


print("\n[preprocess] bilinear upsampling Uc 15x15 -> 41x41 ...")
Uc_grid_all = bilinear_upsample_velocity_all(U_lf_all)
Uf_grid_all = U_hf_all

N_OPERATORS = N_OPERATORS_EXPECTED
N_TIMES = N_TIMES_EXPECTED
N_SIDE = HF_SIDE_EXPECTED
N_GRID = N_SIDE * N_SIDE
NX_XY = 2 * N_GRID
NX_VEL = 2 * N_GRID

X_hf, Y_hf = np.meshgrid(
    x_hf, y_hf, indexing="xy"
)
row_xy_raw = np.concatenate(
    [X_hf.reshape(-1), Y_hf.reshape(-1)],
    axis=0,
).astype(np.float32)

Uc_flat_all = Uc_grid_all.reshape(
    N_OPERATORS, N_TIMES, NX_VEL
).astype(np.float32, copy=False)

Uf_flat_all = Uf_grid_all.reshape(
    N_OPERATORS, N_TIMES, NX_VEL
).astype(np.float32, copy=False)

print("  Uc upsampled                :", Uc_grid_all.shape)
print("  Uf target                   :", Uf_grid_all.shape)
print("  model rows                  : XY / time / Uc / Uf")
print("  beta model input            : NO")
print("  H model input               : NO")
print("  A model input               : NO")


# =============================================================================
# TRAIN-SYSTEM-ONLY NORMALIZATION
# =============================================================================

def velocity_component_stats(
    arr: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    vals = arr.reshape(-1, NX_VEL)
    means = []
    stds = []
    for ci in range(2):
        a = ci * N_GRID
        b = (ci + 1) * N_GRID
        v = vals[:, a:b].astype(
            np.float64, copy=False
        )
        means.append(float(v.mean()))
        stds.append(float(v.std() + 1.0e-6))
    return (
        np.asarray(means, dtype=np.float32),
        np.asarray(stds, dtype=np.float32),
    )


def normalize_velocity(
    arr: np.ndarray,
    means: np.ndarray,
    stds: np.ndarray,
) -> np.ndarray:
    out = np.empty_like(arr, dtype=np.float32)
    out[..., :N_GRID] = (
        arr[..., :N_GRID] - means[0]
    ) / stds[0]
    out[..., N_GRID:] = (
        arr[..., N_GRID:] - means[1]
    ) / stds[1]
    return out


Uc_train_view = Uc_flat_all[train_operator_indices]
Uf_train_view = Uf_flat_all[train_operator_indices]

Uc_comp_mean, Uc_comp_std = velocity_component_stats(
    Uc_train_view
)
Uf_comp_mean, Uf_comp_std = velocity_component_stats(
    Uf_train_view
)

Uc_n_all = normalize_velocity(
    Uc_flat_all,
    Uc_comp_mean,
    Uc_comp_std,
)
Uf_n_all = normalize_velocity(
    Uf_flat_all,
    Uf_comp_mean,
    Uf_comp_std,
)

x_part = row_xy_raw[:N_GRID]
y_part = row_xy_raw[N_GRID:]

xy_comp_mean = np.asarray(
    [x_part.mean(), y_part.mean()],
    dtype=np.float32,
)
xy_comp_std = np.asarray(
    [
        x_part.std() + 1.0e-6,
        y_part.std() + 1.0e-6,
    ],
    dtype=np.float32,
)

xy_n = np.empty_like(row_xy_raw, dtype=np.float32)
xy_n[:N_GRID] = (
    x_part - xy_comp_mean[0]
) / xy_comp_std[0]
xy_n[N_GRID:] = (
    y_part - xy_comp_mean[1]
) / xy_comp_std[1]

# Time is an explicit within-system coordinate, not an OOD variable here.
# Deterministic normalization to [-1, 1].
TIME_MIN = float(time_values.min())
TIME_MAX = float(time_values.max())
time_n = (
    2.0 * (time_values - TIME_MIN)
    / (TIME_MAX - TIME_MIN)
    - 1.0
).astype(np.float32)

Uf_comp_mean_t = torch.tensor(
    Uf_comp_mean,
    dtype=torch.float32,
    device=DEVICE,
)
Uf_comp_std_t = torch.tensor(
    Uf_comp_std,
    dtype=torch.float32,
    device=DEVICE,
)

print("\n[normalization]")
print("  source                    : TRAIN SYSTEMS ONLY for Uc/Uf")
print("  Uc means/stds             :", Uc_comp_mean.tolist(), "/", Uc_comp_std.tolist())
print("  Uf means/stds             :", Uf_comp_mean.tolist(), "/", Uf_comp_std.tolist())
print("  XY means/stds             :", xy_comp_mean.tolist(), "/", xy_comp_std.tolist())
print("  time normalization        : [1,50] -> [-1,1]")


# =============================================================================
# METRICS
# =============================================================================

def summarize_array(
    arr: np.ndarray,
    scale: float = 1.0,
) -> dict:
    x = np.asarray(arr, dtype=np.float64) * scale
    return {
        "mean": float(np.mean(x)),
        "median": float(np.median(x)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
        "std": float(np.std(x)),
        "p90": float(np.percentile(x, 90.0)),
        "p95": float(np.percentile(x, 95.0)),
    }


def rel_l2_np(
    pred: np.ndarray,
    target: np.ndarray,
    eps: float = 1.0e-12,
) -> np.ndarray:
    return np.linalg.norm(
        pred - target, axis=1
    ) / (
        np.linalg.norm(target, axis=1) + eps
    )


def unnormalize_velocity_torch(
    u_n: torch.Tensor,
) -> torch.Tensor:
    u = (
        u_n[:, :N_GRID] * Uf_comp_std_t[0]
        + Uf_comp_mean_t[0]
    )
    v = (
        u_n[:, N_GRID:] * Uf_comp_std_t[1]
        + Uf_comp_mean_t[1]
    )
    return torch.cat([u, v], dim=1)


# =============================================================================
# EPISODE BUILDERS
# =============================================================================

def get_time_arrays(
    op_idx: int,
    time_idx: int,
):
    return (
        float(time_n[time_idx]),
        Uc_n_all[op_idx, time_idx],
        Uf_n_all[op_idx, time_idx],
        Uf_flat_all[op_idx, time_idx],
    )


def build_one_context(
    op_idx: int,
    demo_time_indices,
    query_time_idx: int,
):
    """
    Build one ICL context from ONE physical system.

    Sequence:
        [XY,t,Uc,Uf] x K + [XY,t,Uc]

    Target:
        full Uf at query time
    """
    d_cur = len(demo_time_indices)
    if not (1 <= d_cur <= MAX_DEMOS):
        raise ValueError(
            f"Invalid number of demonstrations: {d_cur}"
        )

    demo_time_indices = np.asarray(
        demo_time_indices,
        dtype=np.int64,
    ).reshape(-1)

    if len(set(map(int, demo_time_indices.tolist()))) != d_cur:
        raise ValueError("Duplicate demonstration times.")

    if int(query_time_idx) in set(
        map(int, demo_time_indices.tolist())
    ):
        raise ValueError(
            "Query time cannot also be a demonstration time."
        )

    S = 4 * d_cur + 3
    mat_ctx = np.zeros(
        (S, NX_VEL),
        dtype=np.float32,
    )
    row_types = np.empty(
        (S,),
        dtype=np.int64,
    )

    k = 0

    for ti in demo_time_indices:
        ti = int(ti)
        t_scalar, uc_n, uf_n, _ = get_time_arrays(
            op_idx, ti
        )

        mat_ctx[k, :] = xy_n
        row_types[k] = TYPE_XY_DEMO
        k += 1

        mat_ctx[k, 0] = t_scalar
        row_types[k] = TYPE_TIME_DEMO
        k += 1

        mat_ctx[k, :] = uc_n
        row_types[k] = TYPE_UC_DEMO
        k += 1

        mat_ctx[k, :] = uf_n
        row_types[k] = TYPE_UF_DEMO
        k += 1

    qt, quc_n, quf_n, quf_phys = get_time_arrays(
        op_idx,
        int(query_time_idx),
    )

    mat_ctx[k, :] = xy_n
    row_types[k] = TYPE_XY_QUERY
    k += 1

    mat_ctx[k, 0] = qt
    row_types[k] = TYPE_TIME_QUERY
    k += 1

    mat_ctx[k, :] = quc_n
    row_types[k] = TYPE_UC_QUERY
    k += 1

    assert k == S

    return (
        mat_ctx,
        row_types,
        quf_n.astype(np.float32, copy=False),
        quf_phys.astype(np.float32, copy=False),
    )


def make_train_batch(
    batch_size: int,
):
    """
    One K per batch so the sequence tensor is rectangular.

    For every item:
      1. sample one TRAIN system;
      2. sample K+1 distinct time indices from its trajectory;
      3. first K sampled indices are demos, last is query.

    Demo order is intentionally random; time is explicitly encoded.
    """
    d_cur = int(
        rng_train.integers(
            MIN_DEMOS,
            MAX_DEMOS + 1,
        )
    )
    S = 4 * d_cur + 3

    mat_ctx = np.zeros(
        (batch_size, S, NX_VEL),
        dtype=np.float32,
    )
    row_type_arr = np.empty(
        (batch_size, S),
        dtype=np.int64,
    )
    target_n = np.empty(
        (batch_size, NX_VEL),
        dtype=np.float32,
    )

    for i in range(batch_size):
        op_idx = int(
            rng_train.choice(
                train_operator_indices
            )
        )
        idxs = rng_train.choice(
            N_TIMES,
            size=d_cur + 1,
            replace=False,
        )

        demos = idxs[:d_cur]
        q_idx = int(idxs[-1])

        ctx_i, type_i, tgt_n_i, _ = (
            build_one_context(
                op_idx,
                demos,
                q_idx,
            )
        )

        mat_ctx[i] = ctx_i
        row_type_arr[i] = type_i
        target_n[i] = tgt_n_i

    return (
        torch.from_numpy(mat_ctx).to(
            DEVICE, non_blocking=True
        ),
        torch.from_numpy(row_type_arr).to(
            DEVICE, non_blocking=True
        ),
        torch.from_numpy(target_n).to(
            DEVICE, non_blocking=True
        ),
    )


def make_fixed_train_monitor_cases(
    n_cases: int,
    seed: int,
    fixed_demos: int,
):
    r = np.random.default_rng(seed)
    cases = []

    for _ in range(n_cases):
        op_idx = int(
            r.choice(train_operator_indices)
        )
        idxs = r.choice(
            N_TIMES,
            size=fixed_demos + 1,
            replace=False,
        )

        cases.append(
            {
                "op_idx": op_idx,
                "demo_indices": idxs[
                    :fixed_demos
                ].astype(np.int64),
                "query_idx": int(idxs[-1]),
                "split": "train_monitor",
            }
        )

    return cases


def make_seeded_heldout_test_partition(
    seed: int,
    k: int,
):
    """
    Build ONE reproducible random support set per held-out ID system.

    No times are hard-coded.
    Different test systems generally receive different support times.
    The remaining 50-k snapshots are all official queries.
    """
    r = np.random.default_rng(seed)

    demo_matrix = np.empty(
        (
            len(test_operator_indices),
            k,
        ),
        dtype=np.int64,
    )

    query_matrix = np.empty(
        (
            len(test_operator_indices),
            N_TIMES - k,
        ),
        dtype=np.int64,
    )

    all_idx = np.arange(
        N_TIMES, dtype=np.int64
    )

    cases = []

    for row, op_idx in enumerate(
        test_operator_indices
    ):
        # choice() preserves a random order, so the support sequence
        # is not forced into chronological order.
        demos = r.choice(
            N_TIMES,
            size=k,
            replace=False,
        ).astype(np.int64)

        keep = np.ones(
            N_TIMES,
            dtype=bool,
        )
        keep[demos] = False
        queries = all_idx[keep]

        demo_matrix[row] = demos
        query_matrix[row] = queries

        for q_idx in queries:
            cases.append(
                {
                    "op_idx": int(op_idx),
                    "test_row": int(row),
                    "demo_indices": demos.copy(),
                    "query_idx": int(q_idx),
                    "split": "heldout_ID_system_test",
                }
            )

    return demo_matrix, query_matrix, cases


train_eval_cases = make_fixed_train_monitor_cases(
    N_TRAIN_EVAL_CASES,
    seed=SEED + 2,
    fixed_demos=TEST_K,
)

(
    test_demo_time_indices,
    test_query_time_indices,
    test_cases,
) = make_seeded_heldout_test_partition(
    TEST_SUPPORT_SEED,
    TEST_K,
)

EXPECTED_TEST_CASES = (
    N_TEST_OPERATORS_EXPECTED
    * (N_TIMES_EXPECTED - TEST_K)
)
assert len(test_cases) == EXPECTED_TEST_CASES

test_partition_path = os.path.join(
    SAVE_DIR,
    f"{RUN_TAG}_test_support_partition.npz",
)

np.savez_compressed(
    test_partition_path,
    seed=np.array(
        TEST_SUPPORT_SEED,
        dtype=np.int64,
    ),
    K=np.array(TEST_K, dtype=np.int64),
    test_operator_indices=test_operator_indices,
    demo_time_indices=test_demo_time_indices,
    demo_times=time_values[
        test_demo_time_indices
    ],
    query_time_indices=test_query_time_indices,
    query_times=time_values[
        test_query_time_indices
    ],
)

print("\n[held-out ID evaluation protocol]")
print("  support partition source    : SEEDED RANDOM IN TRAINING SCRIPT")
print("  support seed                :", TEST_SUPPORT_SEED)
print("  held-out systems            :", len(test_operator_indices))
print("  demos/system                :", TEST_K)
print("  queries/system              :", N_TIMES - TEST_K)
print("  total official queries      :", len(test_cases))
print("  fixed same times for all?   : NO")
print("  temporal OOD                : NO")
print("  forecasting claim           : NO")
print("  first held-out system       :", int(test_operator_indices[0]))
print("  first demo time indices     :", test_demo_time_indices[0].tolist())
print("  first demo actual times     :", time_values[test_demo_time_indices[0]].tolist())
print("  partition saved             :", test_partition_path)


# =============================================================================
# MODEL
# =============================================================================

class RowSequenceSemanticEncoderTransformer(
    nn.Module
):
    def __init__(
        self,
        nx_xy: int,
        nx_vel: int,
        max_ctx_rows: int,
        n_row_types: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_ff: int,
        dropout: float,
    ):
        super().__init__()

        self.nx_xy = nx_xy
        self.nx_vel = nx_vel
        self.max_ctx_rows = max_ctx_rows
        self.d_model = d_model

        self.xy_encoder = nn.Linear(
            nx_xy, d_model
        )
        self.time_encoder = nn.Linear(
            1, d_model
        )

        # Shared Uc/Uf encoder.
        self.velocity_encoder = nn.Linear(
            nx_vel, d_model
        )

        self.row_type_emb = nn.Embedding(
            n_row_types, d_model
        )
        self.pos_emb = nn.Embedding(
            max_ctx_rows, d_model
        )

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            enc_layer,
            num_layers=num_layers,
        )

        def make_head():
            return nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, d_model),
                nn.GELU(),
                nn.Linear(d_model, N_GRID),
            )

        self.u_head = make_head()
        self.v_head = make_head()

        self.register_buffer(
            "pos_ids",
            torch.arange(
                max_ctx_rows,
                dtype=torch.long,
            ),
            persistent=False,
        )

    def forward(
        self,
        mat_ctx: torch.Tensor,
        row_type_arr: torch.Tensor,
    ) -> torch.Tensor:
        B, S, NX_ = mat_ctx.shape

        if NX_ != self.nx_vel:
            raise RuntimeError(
                f"Context width mismatch: {NX_} vs {self.nx_vel}"
            )
        if S > self.max_ctx_rows:
            raise RuntimeError(
                f"Sequence length {S} > max {self.max_ctx_rows}"
            )

        flat_ctx = mat_ctx.reshape(
            B * S, NX_
        )
        flat_types = row_type_arr.reshape(
            B * S
        )

        xy_mask = (
            (flat_types == TYPE_XY_DEMO)
            | (flat_types == TYPE_XY_QUERY)
        )
        time_mask = (
            (flat_types == TYPE_TIME_DEMO)
            | (flat_types == TYPE_TIME_QUERY)
        )
        vel_mask = (
            (flat_types == TYPE_UC_DEMO)
            | (flat_types == TYPE_UF_DEMO)
            | (flat_types == TYPE_UC_QUERY)
        )

        # AMP-safe allocation.
        #
        # Under CUDA autocast, encoder outputs are typically float16 while
        # mat_ctx/flat_ctx remain float32. Therefore h_flat MUST be allocated
        # from an encoded tensor, not from flat_ctx, otherwise indexed
        # assignment fails with:
        #   "Index put requires the source and destination dtypes match"
        #
        # XY rows are always present in every valid context, so use the XY
        # encoder output as the dtype/device template.
        if not torch.any(xy_mask):
            raise RuntimeError("Every context must contain XY rows.")

        xy_encoded = self.xy_encoder(
            flat_ctx[
                xy_mask,
                :self.nx_xy,
            ]
        )

        h_flat = xy_encoded.new_zeros(
            (B * S, self.d_model)
        )
        h_flat[xy_mask] = xy_encoded

        if torch.any(time_mask):
            time_encoded = self.time_encoder(
                flat_ctx[
                    time_mask,
                    :1,
                ]
            )
            h_flat[time_mask] = time_encoded.to(
                dtype=h_flat.dtype
            )

        if torch.any(vel_mask):
            vel_encoded = self.velocity_encoder(
                flat_ctx[
                    vel_mask,
                    :self.nx_vel,
                ]
            )
            h_flat[vel_mask] = vel_encoded.to(
                dtype=h_flat.dtype
            )

        h = h_flat.reshape(
            B, S, self.d_model
        )

        h = (
            h
            + self.row_type_emb(row_type_arr)
            + self.pos_emb(
                self.pos_ids[:S]
            )[None, :, :]
        )

        # No causal mask: all visible demonstrations can interact.
        h = self.encoder(h)

        # Final visible row is Uc_query.
        h_last = h[:, -1, :]

        pred_u = self.u_head(h_last)
        pred_v = self.v_head(h_last)

        return torch.cat(
            [pred_u, pred_v],
            dim=1,
        )


model = RowSequenceSemanticEncoderTransformer(
    nx_xy=NX_XY,
    nx_vel=NX_VEL,
    max_ctx_rows=MAX_CTX_ROWS,
    n_row_types=N_ROW_TYPES,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=D_FF,
    dropout=DROPOUT,
).to(DEVICE)

n_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n[model audit]")
print("  model type                 : RowSequenceSemanticEncoderTransformer")
print("  parameters                 :", f"{n_params:,}")
print("  D_MODEL / heads            :", D_MODEL, "/", N_HEAD)
print("  layers / D_FF              :", N_LAYERS, "/", D_FF)
print("  XY encoder                 : YES")
print("  scalar time encoder        : YES")
print("  shared Uc/Uf encoder       : YES")
print("  beta encoder               : NO")
print("  H encoder                  : NO")
print("  A encoder                  : NO")
print("  causal mask                : NO")
print("  prediction                 : direct full Uf")
print("  residual target            : NO")


# =============================================================================
# LOSS
# =============================================================================

def balanced_component_mse(
    pred_n: torch.Tensor,
    target_n: torch.Tensor,
):
    pred_u = pred_n[:, :N_GRID]
    pred_v = pred_n[:, N_GRID:]
    tgt_u = target_n[:, :N_GRID]
    tgt_v = target_n[:, N_GRID:]

    loss_u = torch.mean(
        (pred_u - tgt_u) ** 2
    )
    loss_v = torch.mean(
        (pred_v - tgt_v) ** 2
    )

    loss = 0.5 * (
        loss_u + loss_v
    )

    return {
        "loss_total": loss,
        "loss_mse": loss.detach(),
        "loss_u": loss_u.detach(),
        "loss_v": loss_v.detach(),
    }


# =============================================================================
# OPTIMIZER / SCHEDULER
# =============================================================================

TOTAL_STEPS = EPOCHS * STEPS_PER_EPOCH
WARMUP_STEPS = max(
    1,
    int(WARMUP_FRAC * TOTAL_STEPS),
)

opt = torch.optim.AdamW(
    model.parameters(),
    lr=MAX_LR,
    weight_decay=WEIGHT_DECAY,
)

warmup_scheduler = (
    torch.optim.lr_scheduler.LinearLR(
        opt,
        start_factor=BASE_LR / MAX_LR,
        end_factor=1.0,
        total_iters=WARMUP_STEPS,
    )
)

cosine_scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=max(
            1,
            TOTAL_STEPS - WARMUP_STEPS,
        ),
        eta_min=MIN_LR,
    )
)

scheduler = (
    torch.optim.lr_scheduler.SequentialLR(
        opt,
        schedulers=[
            warmup_scheduler,
            cosine_scheduler,
        ],
        milestones=[WARMUP_STEPS],
    )
)

if DEVICE == "cuda":
    try:
        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=USE_AMP,
        )

        def amp_context():
            return torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP,
            )

    except Exception:
        scaler = torch.cuda.amp.GradScaler(
            enabled=USE_AMP
        )

        def amp_context():
            return torch.cuda.amp.autocast(
                enabled=USE_AMP
            )
else:
    scaler = None

    def amp_context():
        return nullcontext()


print("\n[optimizer]")
print("  AdamW weight decay         :", WEIGHT_DECAY)
print("  batch size                 :", BATCH_SIZE)
print("  epochs                     :", EPOCHS)
print("  steps / epoch              :", STEPS_PER_EPOCH)
print("  total steps                :", TOTAL_STEPS)
print("  warmup steps               :", WARMUP_STEPS)
print("  LR base/max/min            :", BASE_LR, "/", MAX_LR, "/", MIN_LR)
print("  grad clip                  :", GRAD_CLIP)
print("  AMP                        :", USE_AMP)


# =============================================================================
# EVALUATION
# =============================================================================

def make_eval_batch(
    cases_subset,
):
    d_cur = len(
        cases_subset[0]["demo_indices"]
    )
    S = 4 * d_cur + 3
    B = len(cases_subset)

    mat_ctx = np.zeros(
        (B, S, NX_VEL),
        dtype=np.float32,
    )
    row_types = np.empty(
        (B, S),
        dtype=np.int64,
    )
    target_n = np.empty(
        (B, NX_VEL),
        dtype=np.float32,
    )
    target = np.empty(
        (B, NX_VEL),
        dtype=np.float32,
    )
    op_indices = np.empty(
        (B,),
        dtype=np.int64,
    )
    query_indices = np.empty(
        (B,),
        dtype=np.int64,
    )

    for i, case in enumerate(
        cases_subset
    ):
        op_idx = int(case["op_idx"])
        demos = case["demo_indices"]
        q_idx = int(case["query_idx"])

        c, rt, tn, tp = (
            build_one_context(
                op_idx,
                demos,
                q_idx,
            )
        )

        mat_ctx[i] = c
        row_types[i] = rt
        target_n[i] = tn
        target[i] = tp
        op_indices[i] = op_idx
        query_indices[i] = q_idx

    return (
        torch.from_numpy(mat_ctx).to(
            DEVICE, non_blocking=True
        ),
        torch.from_numpy(row_types).to(
            DEVICE, non_blocking=True
        ),
        torch.from_numpy(target_n).to(
            DEVICE, non_blocking=True
        ),
        torch.from_numpy(target).to(
            DEVICE, non_blocking=True
        ),
        op_indices,
        query_indices,
    )


@torch.no_grad()
def eval_model_cases(
    cases,
    return_predictions: bool = False,
):
    model.eval()

    mse_list = []
    rel_list = []
    r2_list = []
    u_rel_list = []
    v_rel_list = []
    op_list = []
    q_list = []
    pred_list = []
    target_list = []

    for start in range(
        0,
        len(cases),
        EVAL_BATCH_SIZE,
    ):
        subset = cases[
            start:start + EVAL_BATCH_SIZE
        ]

        (
            mat_ctx,
            row_types,
            _,
            target,
            op_idx,
            q_idx,
        ) = make_eval_batch(subset)

        with amp_context():
            pred_n = model(
                mat_ctx,
                row_types,
            )

        pred = unnormalize_velocity_torch(
            pred_n.float()
        )

        err = pred - target

        mse = torch.mean(
            err ** 2,
            dim=1,
        )

        rel = (
            torch.linalg.norm(
                err, dim=1
            )
            / torch.linalg.norm(
                target, dim=1
            ).clamp_min(EPS)
        )

        y_mean = torch.mean(
            target,
            dim=1,
            keepdim=True,
        )
        sse = torch.sum(
            err ** 2,
            dim=1,
        )
        sst = torch.sum(
            (target - y_mean) ** 2,
            dim=1,
        ).clamp_min(EPS)

        r2 = 1.0 - sse / sst

        pu = pred[:, :N_GRID]
        pv = pred[:, N_GRID:]
        tu = target[:, :N_GRID]
        tv = target[:, N_GRID:]

        urel = (
            torch.linalg.norm(
                pu - tu,
                dim=1,
            )
            / torch.linalg.norm(
                tu,
                dim=1,
            ).clamp_min(EPS)
        )

        vrel = (
            torch.linalg.norm(
                pv - tv,
                dim=1,
            )
            / torch.linalg.norm(
                tv,
                dim=1,
            ).clamp_min(EPS)
        )

        mse_list.append(
            mse.cpu().numpy()
        )
        rel_list.append(
            rel.cpu().numpy()
        )
        r2_list.append(
            r2.cpu().numpy()
        )
        u_rel_list.append(
            urel.cpu().numpy()
        )
        v_rel_list.append(
            vrel.cpu().numpy()
        )
        op_list.append(op_idx)
        q_list.append(q_idx)

        if return_predictions:
            pred_list.append(
                pred.cpu().numpy()
            )
            target_list.append(
                target.cpu().numpy()
            )

    mse_arr = np.concatenate(mse_list)
    rel_arr = np.concatenate(rel_list)
    r2_arr = np.concatenate(r2_list)
    urel_arr = np.concatenate(u_rel_list)
    vrel_arr = np.concatenate(v_rel_list)
    op_arr = np.concatenate(op_list)
    q_arr = np.concatenate(q_list)

    per_operator = {}

    for op_idx in np.unique(op_arr):
        mask = op_arr == op_idx

        per_operator[str(int(op_idx))] = {
            "A_rel": float(
                A_rel_values[int(op_idx)]
            ),
            "N_queries": int(
                mask.sum()
            ),
            "RelL2_summary_%": summarize_array(
                rel_arr[mask],
                100.0,
            ),
            "R2_summary": summarize_array(
                r2_arr[mask],
                1.0,
            ),
            "u_RelL2_summary_%": summarize_array(
                urel_arr[mask],
                100.0,
            ),
            "v_RelL2_summary_%": summarize_array(
                vrel_arr[mask],
                100.0,
            ),
        }

    out = {
        "N_cases": int(len(cases)),
        "N_operators": int(
            len(np.unique(op_arr))
        ),
        "MSE_mean": float(
            mse_arr.mean()
        ),
        "RelL2_summary_%": summarize_array(
            rel_arr,
            100.0,
        ),
        "R2_summary": summarize_array(
            r2_arr,
            1.0,
        ),
        "ComponentRelL2_summary_%": {
            "u": summarize_array(
                urel_arr,
                100.0,
            ),
            "v": summarize_array(
                vrel_arr,
                100.0,
            ),
        },
        "BalancedComponentRelL2_mean_%": float(
            0.5
            * (
                100.0 * urel_arr.mean()
                + 100.0 * vrel_arr.mean()
            )
        ),
        "per_operator": per_operator,
    }

    if return_predictions:
        out["predictions"] = np.concatenate(
            pred_list,
            axis=0,
        )
        out["targets"] = np.concatenate(
            target_list,
            axis=0,
        )
        out["rel_per_case"] = rel_arr
        out["r2_per_case"] = r2_arr
        out["operator_indices"] = op_arr
        out["query_time_indices"] = q_arr

    return out


def print_metrics(
    prefix: str,
    metrics: dict,
):
    r = metrics["RelL2_summary_%"]
    u = metrics[
        "ComponentRelL2_summary_%"
    ]["u"]
    v = metrics[
        "ComponentRelL2_summary_%"
    ]["v"]
    r2 = metrics["R2_summary"]

    print(
        f"{prefix}: "
        f"RelL2 mean={r['mean']:.4f}% | "
        f"median={r['median']:.4f}% | "
        f"p90={r['p90']:.4f}% | "
        f"max={r['max']:.4f}% | "
        f"R2 mean={r2['mean']:.6f} | "
        f"MSE={metrics['MSE_mean']:.6e} | "
        f"u={u['mean']:.4f}% | "
        f"v={v['mean']:.4f}%"
    )


# =============================================================================
# RAW LF BASELINE ON EXACT OFFICIAL 1350 QUERIES
# =============================================================================

def raw_uc_test_metrics():
    pred = []
    tgt = []
    ops = []
    qidx = []

    for case in test_cases:
        op_idx = int(case["op_idx"])
        q_idx = int(case["query_idx"])

        pred.append(
            Uc_flat_all[
                op_idx,
                q_idx,
            ]
        )
        tgt.append(
            Uf_flat_all[
                op_idx,
                q_idx,
            ]
        )
        ops.append(op_idx)
        qidx.append(q_idx)

    pred = np.stack(
        pred
    ).astype(np.float32)
    tgt = np.stack(
        tgt
    ).astype(np.float32)

    ops = np.asarray(
        ops,
        dtype=np.int64,
    )
    qidx = np.asarray(
        qidx,
        dtype=np.int64,
    )

    rel = rel_l2_np(
        pred,
        tgt,
    )
    mse = np.mean(
        (pred - tgt) ** 2,
        axis=1,
    )

    per_operator = {}

    for op_idx in np.unique(ops):
        mask = ops == op_idx
        per_operator[str(int(op_idx))] = {
            "A_rel": float(
                A_rel_values[int(op_idx)]
            ),
            "N_queries": int(mask.sum()),
            "RelL2_summary_%": summarize_array(
                rel[mask],
                100.0,
            ),
            "MSE_mean": float(
                mse[mask].mean()
            ),
        }

    return {
        "N_cases": int(len(rel)),
        "RelL2_summary_%": summarize_array(
            rel,
            100.0,
        ),
        "MSE_mean": float(
            mse.mean()
        ),
        "per_operator": per_operator,
    }


raw_uc_test = raw_uc_test_metrics()

print("\n[raw LF -> HF baseline | exact official held-out queries]")
print(
    "  Uc(upscaled) -> Uf RelL2%: "
    f"mean={raw_uc_test['RelL2_summary_%']['mean']:.4f}, "
    f"median={raw_uc_test['RelL2_summary_%']['median']:.4f}, "
    f"p90={raw_uc_test['RelL2_summary_%']['p90']:.4f}, "
    f"max={raw_uc_test['RelL2_summary_%']['max']:.4f}"
)
print(
    f"  MSE mean: {raw_uc_test['MSE_mean']:.6e}"
)


# =============================================================================
# CHECKPOINTS / OUTPUTS
# =============================================================================

final_ckpt_path = os.path.join(
    SAVE_DIR,
    f"{RUN_TAG}_FINAL_fixedbudget.pt",
)
history_path = os.path.join(
    SAVE_DIR,
    f"{RUN_TAG}_history.json",
)
metrics_path = os.path.join(
    SAVE_DIR,
    f"{RUN_TAG}_final_metrics.json",
)
pred_path = os.path.join(
    SAVE_DIR,
    f"{RUN_TAG}_final_test_predictions.npz",
)


def get_rng_state():
    return {
        "python": random.getstate(),
        "numpy_legacy": np.random.get_state(),
        "rng_train": rng_train.bit_generator.state,
        "torch": torch.get_rng_state(),
        "cuda": (
            torch.cuda.get_rng_state_all()
            if DEVICE == "cuda"
            else None
        ),
    }


def restore_rng_state(
    state: dict,
):
    if not state:
        return

    if "python" in state:
        random.setstate(state["python"])
    if "numpy_legacy" in state:
        np.random.set_state(
            state["numpy_legacy"]
        )
    if "rng_train" in state:
        rng_train.bit_generator.state = (
            state["rng_train"]
        )
    if "torch" in state:
        torch.set_rng_state(
            state["torch"]
        )
    if (
        DEVICE == "cuda"
        and state.get("cuda") is not None
    ):
        torch.cuda.set_rng_state_all(
            state["cuda"]
        )


def checkpoint_payload(
    epoch: int,
    history: list,
):
    return {
        "model": model.state_dict(),
        "optimizer": opt.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": (
            scaler.state_dict()
            if scaler is not None
            else None
        ),
        "epoch": int(epoch),
        "global_step": int(
            epoch * STEPS_PER_EPOCH
        ),
        "rng_state": get_rng_state(),
        "normalization": {
            "Uc_comp_mean": Uc_comp_mean,
            "Uc_comp_std": Uc_comp_std,
            "Uf_comp_mean": Uf_comp_mean,
            "Uf_comp_std": Uf_comp_std,
            "xy_comp_mean": xy_comp_mean,
            "xy_comp_std": xy_comp_std,
            "time_min": TIME_MIN,
            "time_max": TIME_MAX,
            "time_formula": "2*(t-time_min)/(time_max-time_min)-1",
        },
        "config": {
            "RUN_TAG": RUN_TAG,
            "NPZ_PATH": NPZ_PATH,
            "DATASET_SHA256": DATASET_SHA256,
            "dataset_version": metadata["version"],
            "protocol": (
                "100 (A,beta) systems x 50 times; "
                "70 train / 30 held-out ID systems; "
                "no temporal OOD"
            ),
            "operator_definition": (
                "operator_j=(A_j,beta_j(x,y))"
            ),
            "train_operator_indices": train_operator_indices.tolist(),
            "test_operator_indices": test_operator_indices.tolist(),
            "A_rel_values": A_rel_values.tolist(),
            "time_values": time_values.tolist(),
            "MIN_DEMOS": MIN_DEMOS,
            "MAX_DEMOS": MAX_DEMOS,
            "TEST_K": TEST_K,
            "TEST_SUPPORT_SEED": TEST_SUPPORT_SEED,
            "BATCH_SIZE": BATCH_SIZE,
            "EPOCHS": EPOCHS,
            "STEPS_PER_EPOCH": STEPS_PER_EPOCH,
            "D_MODEL": D_MODEL,
            "N_HEAD": N_HEAD,
            "N_LAYERS": N_LAYERS,
            "D_FF": D_FF,
            "DROPOUT": DROPOUT,
            "prediction_mode": "direct_full_Uf_no_delta",
            "A_explicit_input": False,
            "beta_model_input": False,
            "H_model_input": False,
            "time_model_input": True,
            "train_episode_rule": (
                "same train system; random K+1 distinct time snapshots"
            ),
            "test_episode_rule": (
                "same held-out ID system; seeded random K support times; "
                "remaining times are queries"
            ),
            "test_checkpoint_selection": (
                "NONE; final fixed-budget only"
            ),
            "lf_velocity_upsampling": (
                "bilinear 15x15->41x41 align_corners=True"
            ),
            "normalization": (
                "Uc/Uf from train systems only; "
                "XY geometric; time [1,50]->[-1,1]"
            ),
            "sequence": (
                "[XY,t,Uc,Uf]xK + [XY,t,Uc] -> Uf_q"
            ),
            "temporal_ood": False,
            "forecasting_task": False,
        },
        "test_support_partition": {
            "seed": TEST_SUPPORT_SEED,
            "demo_time_indices": test_demo_time_indices,
            "query_time_indices": test_query_time_indices,
        },
        "history": history,
    }


def save_checkpoint(
    path: str,
    epoch: int,
    history: list,
):
    torch.save(
        checkpoint_payload(
            epoch,
            history,
        ),
        path,
    )


# =============================================================================
# OPTIONAL RESUME
# =============================================================================

start_epoch = 1
history = []

if RESUME_FROM:
    resume_path = Path(RESUME_FROM)
    if not resume_path.is_file():
        raise FileNotFoundError(
            f"RESUME_FROM does not exist: {resume_path}"
        )

    print("\n[resume] loading:", resume_path)

    ckpt = torch.load(
        str(resume_path),
        map_location=DEVICE,
        weights_only=False,
    )

    ckpt_cfg = ckpt.get(
        "config", {}
    )

    if ckpt_cfg.get(
        "DATASET_SHA256", ""
    ) != DATASET_SHA256:
        raise RuntimeError(
            "Resume checkpoint dataset SHA256 does not match current dataset."
        )

    model.load_state_dict(
        ckpt["model"],
        strict=True,
    )
    opt.load_state_dict(
        ckpt["optimizer"]
    )
    scheduler.load_state_dict(
        ckpt["scheduler"]
    )

    if (
        scaler is not None
        and ckpt.get("scaler") is not None
    ):
        scaler.load_state_dict(
            ckpt["scaler"]
        )

    restore_rng_state(
        ckpt.get("rng_state", {})
    )

    history = ckpt.get(
        "history", []
    )

    completed_epoch = int(
        ckpt.get("epoch", 0)
    )
    start_epoch = completed_epoch + 1

    if start_epoch > EPOCHS:
        raise RuntimeError(
            f"Checkpoint already completed epoch {completed_epoch}, "
            f"but configured EPOCHS={EPOCHS}."
        )

    print(
        f"[resume] completed epoch={completed_epoch}; "
        f"continuing from epoch={start_epoch}"
    )


# =============================================================================
# TRAIN
# =============================================================================

print("\n" + "=" * 128)
print("TRAINING START")
print("=" * 128)
print("Train systems                  :", len(train_operator_indices))
print("Held-out ID test systems       :", len(test_operator_indices))
print("Time snapshots/system          :", N_TIMES)
print("Training K                     :", f"dynamic {MIN_DEMOS}..{MAX_DEMOS}")
print("Testing K                      :", TEST_K)
print("Demo rows                      : [XY,t,Uc,Uf]")
print("Query rows                     : [XY,t,Uc]")
print("A / beta / H supplied          : NO / NO / NO")
print("Target                         : full Uf_q")
print("Temporal OOD                   : NO")
print("Official model selection       : FINAL STEP ONLY")
print("Held-out diagnostics in train  :", RUN_TEST_DIAGNOSTICS)
print("Total scheduled steps          :", TOTAL_STEPS)
print("=" * 128)

start_time = time.time()

for ep in range(
    start_epoch,
    EPOCHS + 1,
):
    model.train()
    epoch_t0 = time.time()

    logs = {
        "loss_total": [],
        "loss_mse": [],
        "loss_u": [],
        "loss_v": [],
    }

    for _ in range(
        STEPS_PER_EPOCH
    ):
        (
            mat_ctx,
            row_types,
            target_n,
        ) = make_train_batch(
            BATCH_SIZE
        )

        opt.zero_grad(
            set_to_none=True
        )

        with amp_context():
            pred_n = model(
                mat_ctx,
                row_types,
            )
            loss_dict = (
                balanced_component_mse(
                    pred_n,
                    target_n,
                )
            )
            loss_total = loss_dict[
                "loss_total"
            ]

        if USE_AMP:
            scaler.scale(
                loss_total
            ).backward()

            scaler.unscale_(opt)

            nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )

            scaler.step(opt)
            scaler.update()

        else:
            loss_total.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )

            opt.step()

        scheduler.step()

        for key in logs:
            logs[key].append(
                float(
                    loss_dict[key]
                    .detach()
                    .cpu()
                )
            )

    epoch_logs = {
        k: float(np.mean(v))
        for k, v in logs.items()
    }

    should_print = (
        ep == 1
        or ep % PRINT_EVERY == 0
        or ep == EPOCHS
    )

    if should_print:
        train_metrics = eval_model_cases(
            train_eval_cases
        )

        lr = float(
            opt.param_groups[0]["lr"]
        )
        elapsed_epoch = (
            time.time() - epoch_t0
        )

        row = {
            "epoch": int(ep),
            "global_step": int(
                ep * STEPS_PER_EPOCH
            ),
            "lr": lr,
            "seconds": float(
                elapsed_epoch
            ),
            "elapsed_total_seconds": float(
                time.time() - start_time
            ),
            **{
                f"train_{k}": v
                for k, v
                in epoch_logs.items()
            },
            "train_monitor": train_metrics,
        }

        print(
            f"epoch {ep:03d}/{EPOCHS} | "
            f"step {ep*STEPS_PER_EPOCH:06d}/{TOTAL_STEPS} | "
            f"loss={epoch_logs['loss_total']:.6e} | "
            f"trainRel={train_metrics['RelL2_summary_%']['mean']:.4f}% | "
            f"lr={lr:.3e} | "
            f"{elapsed_epoch:.1f}s"
        )

        if (
            RUN_TEST_DIAGNOSTICS
            and (
                ep % TEST_DIAGNOSTIC_EVERY == 0
                or ep == EPOCHS
            )
        ):
            test_diag = eval_model_cases(
                test_cases
            )

            row[
                "heldout_ID_test_diagnostic_only"
            ] = test_diag

            print_metrics(
                "  HELD-OUT ID TEST DIAGNOSTIC ONLY",
                test_diag,
            )

        history.append(row)

        with open(
            history_path,
            "w",
        ) as f:
            json.dump(
                history,
                f,
                indent=2,
            )

    if (
        SAVE_PERIODIC_EVERY
        and ep % SAVE_PERIODIC_EVERY == 0
        and ep != EPOCHS
    ):
        periodic_path = os.path.join(
            SAVE_DIR,
            f"{RUN_TAG}_epoch{ep:03d}_resume.pt",
        )

        save_checkpoint(
            periodic_path,
            ep,
            history,
        )

        print(
            "  [resume checkpoint]",
            periodic_path,
        )


# =============================================================================
# OFFICIAL FINAL EVALUATION
# =============================================================================

save_checkpoint(
    final_ckpt_path,
    EPOCHS,
    history,
)

final_train_metrics = eval_model_cases(
    train_eval_cases
)

final_test = eval_model_cases(
    test_cases,
    return_predictions=True,
)

pred = final_test.pop(
    "predictions"
)
target = final_test.pop(
    "targets"
)
rel_per_case = final_test.pop(
    "rel_per_case"
)
r2_per_case = final_test.pop(
    "r2_per_case"
)
op_per_case = final_test.pop(
    "operator_indices"
)
query_idx_per_case = final_test.pop(
    "query_time_indices"
)

op_to_test_row = {
    int(op): row
    for row, op
    in enumerate(
        test_operator_indices.tolist()
    )
}

demo_matrix_per_prediction = np.stack(
    [
        test_demo_time_indices[
            op_to_test_row[int(op)]
        ]
        for op in op_per_case
    ],
    axis=0,
).astype(np.int64)

np.savez_compressed(
    pred_path,
    pred=pred.astype(np.float32),
    target=target.astype(np.float32),
    rel_l2=rel_per_case.astype(np.float32),
    r2=r2_per_case.astype(np.float32),
    operator_indices=op_per_case.astype(np.int64),
    A_rel=A_rel_values[
        op_per_case
    ].astype(np.float32),
    query_time_indices=query_idx_per_case.astype(np.int64),
    query_times=time_values[
        query_idx_per_case
    ].astype(np.float32),
    demo_time_indices=demo_matrix_per_prediction,
    demo_times=time_values[
        demo_matrix_per_prediction
    ].astype(np.float32),
    test_support_seed=np.array(
        TEST_SUPPORT_SEED,
        dtype=np.int64,
    ),
)

final_report = {
    "run_tag": RUN_TAG,
    "dataset": NPZ_PATH,
    "dataset_sha256": DATASET_SHA256,
    "protocol": {
        "dataset": (
            "100 (A,beta) systems x 50 time snapshots"
        ),
        "operator_definition": (
            "operator_j=(A_j,beta_j(x,y))"
        ),
        "train_systems": int(
            len(train_operator_indices)
        ),
        "heldout_ID_test_systems": int(
            len(test_operator_indices)
        ),
        "train_operator_indices": (
            train_operator_indices.tolist()
        ),
        "test_operator_indices": (
            test_operator_indices.tolist()
        ),
        "time_values": (
            time_values.tolist()
        ),
        "train_K": (
            f"dynamic {MIN_DEMOS}..{MAX_DEMOS}"
        ),
        "test_K": TEST_K,
        "test_support_seed": TEST_SUPPORT_SEED,
        "queries_per_test_system": (
            N_TIMES - TEST_K
        ),
        "official_test_queries": int(
            len(test_cases)
        ),
        "test_demonstrations": (
            "seeded-random 5 time snapshots per held-out ID system; "
            "different systems generally have different support times"
        ),
        "query_Uf_in_prompt": False,
        "A_model_input": False,
        "beta_model_input": False,
        "H_model_input": False,
        "time_model_input": True,
        "prediction": "direct full Uf",
        "residual_delta_training": False,
        "official_checkpoint": (
            "final fixed-budget checkpoint"
        ),
        "test_used_for_checkpoint_selection": False,
        "temporal_ood": False,
        "forecasting_task": False,
        "normalization": (
            "velocity statistics from train systems only"
        ),
        "sequence": (
            "[XY,t,Uc,Uf]xK + [XY,t,Uc] -> Uf_q"
        ),
    },
    "model_parameters": int(n_params),
    "model_config": {
        "D_MODEL": D_MODEL,
        "N_HEAD": N_HEAD,
        "N_LAYERS": N_LAYERS,
        "D_FF": D_FF,
        "DROPOUT": DROPOUT,
    },
    "training_config": {
        "BATCH_SIZE": BATCH_SIZE,
        "EPOCHS": EPOCHS,
        "STEPS_PER_EPOCH": STEPS_PER_EPOCH,
        "TOTAL_STEPS": TOTAL_STEPS,
        "BASE_LR": BASE_LR,
        "MAX_LR": MAX_LR,
        "MIN_LR": MIN_LR,
        "WARMUP_FRAC": WARMUP_FRAC,
        "WEIGHT_DECAY": WEIGHT_DECAY,
        "GRAD_CLIP": GRAD_CLIP,
        "RUN_TEST_DIAGNOSTICS": (
            RUN_TEST_DIAGNOSTICS
        ),
    },
    "raw_Uc_test": raw_uc_test,
    "final_train_monitor": final_train_metrics,
    "final_heldout_ID_system_test": final_test,
}

with open(
    metrics_path,
    "w",
) as f:
    json.dump(
        final_report,
        f,
        indent=2,
    )

print("\n" + "=" * 128)
print("OFFICIAL FINAL RESULT | HELD-OUT ID (A,beta) SYSTEMS | FINAL FIXED-BUDGET CHECKPOINT")
print("=" * 128)

print_metrics(
    "FINAL HELD-OUT ID SYSTEM TEST",
    final_test,
)

print(
    "RAW Uc TEST: "
    f"RelL2 mean={raw_uc_test['RelL2_summary_%']['mean']:.4f}% | "
    f"MSE={raw_uc_test['MSE_mean']:.6e}"
)

print("\nPer held-out system mean RelL2%:")

for op_key, d in (
    final_test["per_operator"].items()
):
    row = op_to_test_row[
        int(op_key)
    ]

    print(
        f"  system={int(op_key):03d} | "
        f"A_rel={d['A_rel']:.6f} | "
        f"demos_t={time_values[test_demo_time_indices[row]].astype(int).tolist()} | "
        f"N={d['N_queries']:2d} | "
        f"mean={d['RelL2_summary_%']['mean']:.4f}% | "
        f"median={d['RelL2_summary_%']['median']:.4f}%"
    )

print("\nSaved:")
print("  final checkpoint :", final_ckpt_path)
print("  history          :", history_path)
print("  final metrics    :", metrics_path)
print("  predictions      :", pred_path)
print("  test partition   :", test_partition_path)
print("=" * 128)


ICE-SHEET LEVEL-2 ICL | (A,beta) SYSTEMS x TIME | NO TEMPORAL OOD
Device                    : cuda
Dataset root              : /kaggle/input/notebooks/xli46913/ice-sheet-vars-dataset
Dataset                   : /kaggle/input/notebooks/xli46913/ice-sheet-vars-dataset/ice_icl_Abeta100_time1to50_fullfield_noFixedTimeSplit/ice_icl_Abeta100_time1to50_fullfield_noFixedTimeSplit_master.npz
Dataset SHA256            : 0f70920bbbef146cb213d5566a6bde837a9563457c3db8da2bc9181c3f0672c6
Save dir                  : /kaggle/working/ICE_ICL_Abeta100_time1to50_70train30test_XYtUcUf_sharedVelEnc_d512_L5_FF1536_K1to5_v1_ampfix
System/operator           : (A, beta)
Within-system axis        : time t=1..50
Model sequence            : [XY,t,Uc,Uf]xK + [XY,t,Uc] -> Uf_q
A / beta / H model input  : NO / NO / NO
Temporal OOD              : NO
Official checkpoint       : FINAL FIXED-BUDGET ONLY

[dataset audit]
  metadata version             : ICE_LEVEL2_Abeta100_time1to50_fullfield_noFixedTimeSplit_v2
  beta_g

/tmp/ipykernel_22/2067909619.py:1225: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(



[optimizer]
  AdamW weight decay         : 0.0002
  batch size                 : 64
  epochs                     : 180
  steps / epoch              : 600
  total steps                : 108000
  warmup steps               : 8640
  LR base/max/min            : 1.5e-05 / 5e-05 / 1e-06
  grad clip                  : 1.0
  AMP                        : True

[raw LF -> HF baseline | exact official held-out queries]
  Uc(upscaled) -> Uf RelL2%: mean=30.6168, median=30.8540, p90=33.2703, max=37.3839
  MSE mean: 3.296399e+01

TRAINING START
Train systems                  : 70
Held-out ID test systems       : 30
Time snapshots/system          : 50
Training K                     : dynamic 1..5
Testing K                      : 5
Demo rows                      : [XY,t,Uc,Uf]
Query rows                     : [XY,t,Uc]
A / beta / H supplied          : NO / NO / NO
Target                         : full Uf_q
Temporal OOD                   : NO
Official model selection       : FINAL STEP ONLY
Held-out 